# SIH26146 — Kaggle T4 x2 GNN Training (M3_GCN 38→64→32)

**Goal:** Train `M3_GCN 38→64→32` on real graph+features to replace the 73B stub `models/gnn.pt` and make the hybrid `0.4/0.6` (ensemble fuse) honest.

### Kaggle Settings (do this before Run)
- **Settings → Accelerator:** `GPU T4 x2` (16 GB x2, not P100)
- **Settings → Internet:** `ON` (needs pip + git clone)
- **Quota:** 30h/week, **9h/session** — this notebook trains 200 epochs ≈ 15–25 min on T4
- **Environment:** Python 3.11, torch 2.4.0+cu121, CUDA 12.1

### Download after training
- Output saved to `/kaggle/working/gnn_t4.pt` — **Download** from Output → Files → `gnn_t4.pt`
- Bundle back locally: `cp ~/Downloads/gnn_t4.pt models/gnn.pt && bash scripts/train_gnn_kaggle.sh && make bundle`

### Offline fallback
- Notebook works with **two data sources**: Option A `git clone` (needs Internet ON) and Option B `/kaggle/input/elliptic/*` (Kaggle Dataset, no Internet needed once attached)
- If T4 quota exceeded → change Accelerator to `None` (CPU) or run locally: `uv run python ml/train_gnn.py --train --out /tmp/gnn_smoke.pt`

---
**Cells: 1=pip+CUDA  2=Upload data (git + kaggle/input fallback)  3=Verify+features  4=Train GNN  5=Calibrate+Eval  6=Save for download**

In [ ]:
import logging, sys
for h in logging.root.handlers[:]: logging.root.removeHandler(h)
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
print("LOGS TEST: if you see this, logs work", flush=True)
print("CELL1 pip start verbose (modern PyG 2.3+ no extra wheels)", flush=True)
# Modern PyG: pip install torch_geometric (no -f data.pyg.org, now uses pyg-lib)
# Robust check without shell pipefail issue (was | head || pip never triggered)
import importlib.util, subprocess
if importlib.util.find_spec("torch_geometric") is None:
    print("torch_geometric not found — installing ...", flush=True)
    subprocess.run(["pip", "install", "-q", "torch_geometric"], check=False)
    print("pip install torch_geometric done", flush=True)
else:
    import torch_geometric
    print(f"pyg already installed {torch_geometric.__version__}", flush=True)
print("pip check done", flush=True)
# Light deps (also check via python, not shell ||)
for pkg in ["polars", "duckdb", "networkx", "scikit-learn", "tqdm"]:
    mod = "sklearn" if pkg=="scikit-learn" else pkg.replace("-", "_")
    if importlib.util.find_spec(mod) is None:
        print(f"installing {pkg} ...", flush=True)
        subprocess.run(["pip", "install", "-q", pkg], check=False)
print("deps done", flush=True)
import torch
print(f"torch={torch.__version__} cuda={torch.version.cuda} cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)} | count={torch.cuda.device_count()} (T4 x2 best: use single GPU, not DDP for 50K)")
    import subprocess as sp
    sp.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"], check=False)
else:
    print("CUDA not available — fallback CPU")
import polars as pl, duckdb, networkx as nx, sklearn
print(f"polars={pl.__version__} duckdb={duckdb.__version__} sklearn={sklearn.__version__}")


### Cell2: Upload data — Option A (git clone) + Option B (Kaggle Dataset fallback)
**Pick one:** If Internet ON, git clone works. If quota blocks Internet or repo private, attach Kaggle Dataset `elliptic` containing `data/raw/synthetic/` + `data/graph/` and use `/kaggle/input/` path.


In [ ]:
print("=== CELL 3 START ===", flush=True)
# Cell2: Upload data — Option A git clone, Option B kaggle datasets fallback
import os, pathlib, subprocess, sys
!git clone https://github.com/BlackPool25/bitcoin-sih26146.git /tmp/bitcoin-sih26146 2>&1 | tail -n 5 || echo "git clone skipped (already cloned)"
!ls -lh /tmp/bitcoin-sih26146/ 2>&1 | head -n 20
# Copy all relevant files (backend, ml, scripts, etc.) — was only ml+data, missing backend
!rsync -a /tmp/bitcoin-sih26146/backend /tmp/bitcoin-sih26146/ml /tmp/bitcoin-sih26146/scripts /tmp/bitcoin-sih26146/pyproject.toml /tmp/bitcoin-sih26146/README.md  /kaggle/working/ 2>&1 | tail -n 20 || echo "rsync copy done"
!cp -r /tmp/bitcoin-sih26146/data  /kaggle/working/ 2>&1 | tail -n 5; echo "copied data" 
!ls -lh /kaggle/working/backend 2>&1 | head -n 10 || echo "no backend yet"
!ls -lh /kaggle/working/ml 2>&1 | head -n 10
for cand in ["/kaggle/working", "/tmp/bitcoin-sih26146"]:
    if os.path.exists(os.path.join(cand, "ml/train_gnn.py")):
        os.chdir(cand)
        print(f"%cd {cand}")
        break
print(f"cwd={os.getcwd()}")
!ls -lh data/raw/synthetic data/graph 2>&1 | head -n 20 || echo "no synthetic/graph — will use fallback or build"
!ls -lh /kaggle/input 2>&1 | head -n 20 || echo "no /kaggle/input"
!ls -R data 2>&1 | head -n 80 || echo "data missing"


### Cell3: Verify data + build features if needed
Checks `data/raw/synthetic/synth_50k.csv`, `data/graph/duck.db`, `data/features/features.parquet`. Builds missing artifacts via `ml/features.py` and `ml/train.py`.


In [ ]:
print("=== CELL 5 START ===", flush=True)
# Cell3: Verify data + build features if needed
!ls -lh data/raw/synthetic/synth_50k.csv data/graph/duck.db data/graph/nodes.parquet data/graph/edges.parquet data/features/features.parquet 2>&1 | head -n 30
!cat data/raw/synthetic/synth_50k_meta.json 2>&1 | head -n 20 || echo "no meta json (ok)"
!python -c "import polars as pl; df=pl.read_parquet('data/features/features.parquet'); print(f'features {df.height}x{df.width}', df.columns[:5])" 2>&1 | head -n 20 || echo "features.parquet missing — will build"

# Build features if missing (uses ml/features.py 38 frozen features)
!python ml/features.py --graph data/graph --out data/features 2>&1 | tail -n 20
!ls -lh data/features/features.parquet data/features/feature_names.json 2>&1 | head -n 10

# Train IF as prerequisite for hybrid ensemble (optional but ensures calibrator works)
!python ml/train.py --features data/features/features.parquet --out models/if.pkl 2>&1 | tail -n 20
!ls -lh models/if.pkl 2>&1 | head -n 5

### Cell4: Train GNN (M3_GCN 38→64→32, 200 epochs, T4 x2)
Uses CUDA if available (`--device cuda`), falls back to CPU/synthetic edges via `_build_edge_index` when `duck.db` absent. Logs per-epoch loss, saves best `gnn.pt` (1–5 MB), early stopping if overfit.


In [ ]:
print("=== CELL 7 START ===", flush=True)
# Cell4: Train GNN — 38→64→32 GCN, 200 epochs, early stopping, T4 x2
import os, pathlib
os.environ.setdefault("TORCH_BLAS_PREFER_HIPBLASLT", "0")
print(f"TORCH_BLAS_PREFER_HIPBLASLT={os.environ.get('TORCH_BLAS_PREFER_HIPBLASLT')}")
import os; os.environ["PYDEVD_DISABLE_FILE_VALIDATION"]="1"
!python -Xfrozen_modules=off -u ml/train_gnn.py --train --features data/features/features.parquet --out models/gnn.pt --edge_db data/graph/duck.db 2>&1 | tee /tmp/train.log | tail -n 100
!cat /tmp/train.log | grep -E "epoch|Starting|trained" | tail -n 40
!ls -lh models/gnn.pt && python -c "import pathlib; p=pathlib.Path('models/gnn.pt'); s=p.stat().st_size; print(f'size={s} bytes {s/1e6:.2f}MB range_1-5MB={1_000_000 < s < 5_000_000}')" 2>&1 | tail -n 20
# Inspect checkpoint cleanly (no multiline !python -c)
import pathlib, torch
p=pathlib.Path("models/gnn.pt")
ckpt=torch.load(str(p), map_location="cpu", weights_only=True)
print(f"torch.load ok {type(ckpt)} keys={list(ckpt.keys())[:4] if isinstance(ckpt, dict) else "non-dict"}")
if isinstance(ckpt, dict):
    print(f"config {ckpt.get("config")}")
    sd=ckpt.get("state_dict", {})
    print(f"state_dict total={len(sd)} keys={list(sd.keys())[:4]}")
    for k,v in list(sd.items())[:4]:
        print(k, tuple(v.shape) if hasattr(v,"shape") else type(v))
from ml.train_gnn import _build_edge_index
ei=_build_edge_index(256, "data/graph/duck.db")
print(f"edge_index shape={ei.shape if hasattr(ei, "shape") else type(ei)} fallback={ei.shape[1]>256 if hasattr(ei, "shape") else False}")
!echo "train done — if overfit, reduce epochs" 


### Cell5: Calibrate + Eval (PR-AUC 0.65–0.75 expected vs 0.51 before)
Runs `ml/calibrate.py` (Platt+Isotonic) and `scripts/eval/pr.py --split dfrws` (70/30 temporal+graph-disjoint). Verifies `fuse(0.6,0.8)==0.72` for the 0.4/0.6 ensemble.


In [ ]:
# Cell5: Calibrate + Eval — expect pr_auc 0.65-0.75 vs 0.51 (stub)
!python ml/calibrate.py 2>&1 | tail -n 30
!cat data/eval/calibration.json 2>&1 | head -n 30 || echo "calibration.json not yet — will be after pr.py"

!python scripts/eval/pr.py --split dfrws --out data/eval/pr.json 2>&1 | tail -n 40
!cat data/eval/pr.json 2>&1 | python -m json.tool | head -n 50
!cat data/eval/pr.json | python -c "import json,sys; d=json.load(open('data/eval/pr.json')); print(f\"pr_auc={d.get('pr_auc'):.4f} ece={d.get('ece'):.4f} fpr_at_90={d.get('fpr_at_90_tpr'):.4f} expect 0.65-0.75 vs 0.51 stub\")" 2>&1 | tail -n 10
# jq alternative (if jq installed)
!cat data/eval/pr.json | jq .pr_auc 2>&1 | head -n 5 || echo "jq not installed — used python above"

# Verify ensemble fuse 0.4/0.6
!python ml/ensemble.py --check 2>&1 | tail -n 10
!python -c "from ml.ensemble import fuse; print(f'fuse(0.6,0.8)={fuse(0.6,0.8)} expect 0.72'); assert abs(float(fuse(0.6,0.8))-0.72)<1e-9, 'fuse broken'" 2>&1 | tail -n 5

# Full eval bundle (optional): stress + sigma sweep
!python scripts/eval/stress.py --inject 200 --out data/eval/stress.json 2>&1 | tail -n 10 || echo "stress.py optional"
!python scripts/eval/sigma_sweep.py --sigmas 5,30,120 --out data/eval/sigma_sweep.json 2>&1 | tail -n 10 || echo "sigma_sweep optional"

### Cell6: Save for download + bundle back
Copies `models/gnn.pt` to `/kaggle/working/gnn_t4.pt` (Kaggle Output). Download via **Output → Files → gnn_t4.pt**. Local: `cp gnn_t4.pt models/gnn.pt && bash scripts/train_gnn_kaggle.sh && make bundle`.


In [ ]:
print("=== CELL 11 START ===", flush=True)
# Cell6: Save for download — /kaggle/working/gnn_t4.pt + sha256
!cp models/gnn.pt /kaggle/working/gnn_t4.pt && echo "copied to /kaggle/working/gnn_t4.pt"
!ls -lh /kaggle/working/gnn_t4.pt models/gnn.pt 2>&1 | head -n 10
!sha256sum /kaggle/working/gnn_t4.pt 2>&1 | head -n 5
!sha256sum models/gnn.pt 2>&1 | head -n 5

# Also save calibrator + pr.json alongside for traceability
!cp models/calibrator.pkl /kaggle/working/calibrator_t4.pkl 2>&1 | tail || echo "no calibrator.pkl yet"
!cp data/eval/pr.json /kaggle/working/pr_t4.json 2>&1 | tail || echo "no pr.json yet"
!ls -lh /kaggle/working/ 2>&1 | head -n 30

# Reproducibility snapshot
!pip freeze | grep -E "torch|pyg|polars|duckdb|scikit" | head -n 20
!python -c "import torch; print(f'torch {torch.__version__} cuda={torch.cuda.is_available()}')" 2>&1 | tail -n 5

print("\n=== DONE ===")
print("Download: Kaggle Output panel → Files → gnn_t4.pt (also calibrator_t4.pkl, pr_t4.json)")
print("Local bundle: cp ~/Downloads/gnn_t4.pt models/gnn.pt && bash scripts/train_gnn_kaggle.sh && make bundle")
print("Verify local: uv run python ml/ensemble.py --check  # expect fuse(0.6,0.8)==0.72")
# Fix Kaggle debugger frozen modules warning
import os; os.environ["PYDEVD_DISABLE_FILE_VALIDATION"]="1"
print("Fallback CPU: uv run python ml/train_gnn.py --train --out /tmp/gnn_smoke.pt && ls -lh /tmp/gnn_smoke.pt")